our first step in preparing the data would be to merge the datasets , adding a new column called "new" signaling if a car is new(yes) or used(no) at the same time , we're gonna choose which columns to keep , process which helped decide the ones to keep was the previous crisp phase , main reasons for eliminations were not enough data in the features , or insiginoficant/irrelevant data ..etc



we'll obviously keep pricing, brand , model , kilometrage(which is gonna be 0 for new cars) , body type , fuel type , and seats(theres doors but it's kinda redundant with this) , transmission, the new flag which will be added , i've decided to ditch puissance en chevaux cause it's lacking a lot in used cars ,instead i'll be using puissance fiscale , ill be keeping date mise en circulation (which will be 2025 for the new cars ) , i'll be keeping climatisation(auto or manual) , car dimensions will be ditched as thats already incorporated in body type , fuel usage and everything motor related is also ditched cause ill be keeping puissance fiscale which is directly linked with those .
in short here's how it looks : 

- Price (convert new cars to numeric)
- Brand
- Model
- Kilométrage (0 for new cars)
- Body type (Carrosserie)
- Fuel type (Energie)
- Seats (Nombre de places)
- Transmission (Boîte/Boite vitesse)
- Puissance fiscale (fiscal power)
- Date mise en circulation (2025 for new cars)
- Climatisation (auto/manual)
- New flag (yes/no)



In [307]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load data
df_new = pd.read_csv("../data_scraper/data_scraped/automobileTnNew.csv")
df_used = pd.read_csv("../data_scraper/data_scraped/automobileTnUsed.csv")

def clean_year_string(year_val):
    if pd.isna(year_val):
        return None
    # Convert to string and handle float representation
    year_str = str(year_val)
    # If it looks like a float (has dot and digits), fix it
    if '.' in year_str:
        parts = year_str.split('.')
        if len(parts) == 2:
            month = parts[0]
            year_part = parts[1]
            if len(year_part) == 3:
                year_part = year_part + '0' 
            return f"{month}.{year_part}"
    return year_str

df_used['Mise en circulation'] = df_used['Mise en circulation'].apply(clean_year_string)

# STEP 1: Add the 'new' column to each dataset
df_used['new'] = 'no'
df_new['new'] = 'yes'

# STEP 2: Prepare new cars dataset
df_new['Price_clean'] = df_new['Price'].str.replace(' ', '').astype(float)

# Process climatisation for new cars
def check_auto_climatisation(clim_str):
    if pd.isna(clim_str):
        return 'No'
    clim_str = str(clim_str).lower()
    if 'automatique' in clim_str:
        return 'Yes'
    else:
        return 'no'

df_new['Climatisation_auto'] = df_new['Climatisation'].apply(check_auto_climatisation)

# Add missing columns for new cars with default values
df_new['Kilométrage'] = 0  
df_new['Mise en circulation'] = '12.2025' 

# Select and rename columns for new cars
df_new_clean = df_new[['Brand', 'Model', 'Price_clean', 'Kilométrage', 'Carrosserie', 
                      'Energie', 'Nombre de places', 'Boîte', 'Puissance fiscale', 
                      'Mise en circulation', 'Climatisation_auto', 'new']].copy()

df_new_clean = df_new_clean.rename(columns={
    'Brand': 'brand',
    'Model': 'model', 
    'Price_clean': 'price',
    'Kilométrage': 'kilometrage',
    'Carrosserie': 'body_type',
    'Energie': 'fuel_type',
    'Nombre de places': 'seats',
    'Boîte': 'transmission',
    'Puissance fiscale': 'puissance_fiscale',
    'Mise en circulation': 'registration_year',
    'Climatisation_auto': 'climatisation'
})

# STEP 3: Prepare used cars dataset
# Select and rename columns for used cars
df_used_clean = df_used[['Spécifications - Marque', 'Spécifications - Modèle', 'Price', 
                        'Kilométrage', 'Carrosserie', 'Motorisation - Énergie',
                        'Spécifications - Nombre de places', 'Boite vitesse', 'Puissance fiscale',
                        'Mise en circulation', 'Fonctionnels - Climatisation automatique', 'new']].copy()

df_used_clean = df_used_clean.rename(columns={
    'Spécifications - Marque': 'brand',
    'Spécifications - Modèle': 'model',
    'Price': 'price', 
    'Kilométrage': 'kilometrage',
    'Carrosserie': 'body_type',
    'Motorisation - Énergie': 'fuel_type',
    'Spécifications - Nombre de places': 'seats',
    'Boite vitesse': 'transmission',
    'Puissance fiscale': 'puissance_fiscale',
    'Mise en circulation': 'registration_year',
    'Fonctionnels - Climatisation automatique': 'climatisation'
})

# STEP 4: Merge the datasets
merged_df = pd.concat([df_used_clean, df_new_clean], ignore_index=True)

# STEP 5: Display results
print(f"Final merged dataset shape: {merged_df.shape}")
print(f"Used cars: {(merged_df['new'] == 'no').sum()}")
print(f"New cars: {(merged_df['new'] == 'yes').sum()}")

# VERIFY THE FIX
print(f"\n=== REGISTRATION YEAR DISTRIBUTION (FIXED) ===")
print("Top 20 registration_year values:")
print(merged_df['registration_year'].value_counts().head(20))

print("\nFirst 15 rows of merged dataset:")
print(merged_df.head(15))

Final merged dataset shape: (2658, 12)
Used cars: 2148
New cars: 510

=== REGISTRATION YEAR DISTRIBUTION (FIXED) ===
Top 20 registration_year values:
12.2025    510
5.2022      37
11.2021     36
3.2021      34
1.2021      33
9.2020      30
9.2021      28
3.2022      28
1.2022      27
1.2020      27
6.2021      27
2.2021      26
6.2022      26
7.2020      25
7.2021      25
10.2021     24
11.2019     24
10.2020     24
12.2020     23
8.2021      22
Name: registration_year, dtype: int64

First 15 rows of merged dataset:
            brand         model     price kilometrage body_type  \
0             GWM  Haval Jolion   78000.0  113 000 km       SUV   
1            Audi            A6   75000.0  160 000 km   Berline   
2   Mercedes-Benz     GLE Coupé  460000.0   15 000 km       SUV   
3   Mercedes-Benz      Classe E  118000.0  140 000 km   Berline   
4       Chevrolet        Groove   68000.0   67 000 km       SUV   
5           Skoda         Fabia   47000.0  115 000 km  Citadine   
6        

now we've successfully kept the important columns while fixing some conflicts , next we'll be removing unnecessary rows , for example the overwhelming majority of cars have either diesel or essence, theres also a deecent nmber of orelectric and hybride as fuel type so we'll only be keeping those , also we'll be eliminating cars/brands that have very low represantation in our data well do the same with body type too , we will also remove the outliers when it comes to pricing  and year and mileage 

In [308]:
# BLOCK: DATA CLEANING - ROW FILTERING

print(f"Initial dataset shape: {merged_df.shape}")

# Create a clean copy
merged_clean = merged_df.copy()

# 1. Fuel Type Transformation - simplify categories and remove Compacte
fuel_counts = merged_clean['fuel_type'].value_counts()
print("\nFuel type distribution:")
print(fuel_counts)

# Remove the "Compacte" row first
merged_clean = merged_clean[merged_clean['fuel_type'] != 'Compacte'].copy()

# Transform fuel types into simplified categories
def simplify_fuel_type(fuel):
    fuel = str(fuel).lower()
    if 'electrique' in fuel:
        return 'electrique'
    elif 'hybride' in fuel:
        return 'hybride' 
    else:
        return fuel  # keep essence/diesel as is

merged_clean['fuel_type_simple'] = merged_clean['fuel_type'].apply(simplify_fuel_type)

# Keep all simplified fuel types (essence, diesel, hybride, electrique)
fuel_types_to_keep = ['essence', 'diesel', 'hybride', 'electrique']
merged_clean = merged_clean[merged_clean['fuel_type_simple'].isin(fuel_types_to_keep)].copy()
print(f"After fuel filtering: {merged_clean.shape}")

print("\nSimplified fuel type distribution:")
print(merged_clean['fuel_type_simple'].value_counts())

# 2. Brand Filtering - remove brands with low representation
brand_counts = merged_clean['brand'].value_counts()
print(f"\nBrand counts:")
print(brand_counts)

# Keep brands with at least 10 cars
brands_to_keep = brand_counts[brand_counts >= 10].index
merged_clean = merged_clean[merged_clean['brand'].isin(brands_to_keep)].copy()
print(f"After brand filtering: {merged_clean.shape}")

# 3. Body Type Filtering - remove rare body types
body_counts = merged_clean['body_type'].value_counts()
print(f"\nBody type counts:")
print(body_counts)

# Keep body types with at least 5 cars
bodies_to_keep = body_counts[body_counts >= 30].index
merged_clean = merged_clean[merged_clean['body_type'].isin(bodies_to_keep)].copy()
print(f"After body type filtering: {merged_clean.shape}")

# 4. Price Outlier Removal
price_stats = merged_clean['price'].describe()
print(f"\nPrice statistics before:")
print(price_stats)

# Remove extreme price outliers (bottom 1% and top 1%)
Q1_price = merged_clean['price'].quantile(0.01)
Q3_price = merged_clean['price'].quantile(0.99)
merged_clean = merged_clean[(merged_clean['price'] >= Q1_price) & (merged_clean['price'] <= Q3_price)].copy()
print(f"After price filtering: {merged_clean.shape}")


Initial dataset shape: (2658, 12)

Fuel type distribution:
Essence                           1707
Diesel                             557
Hybride rechargeable essence       127
Electrique                          95
Hybride léger essence               60
Essence | Hybride rechargeable      34
Essence | Hybride léger             25
Essence | Hybride                   23
Hybride essence                     14
Hybride léger diesel                10
Hybride rechargeable diesel          3
Diesel | Hybride léger               2
Compacte                             1
Name: fuel_type, dtype: int64
After fuel filtering: (2657, 13)

Simplified fuel type distribution:
essence       1707
diesel         557
hybride        298
electrique      95
Name: fuel_type_simple, dtype: int64

Brand counts:
Mercedes-Benz    397
Volkswagen       204
KIA              166
BMW              151
Peugeot          141
                ... 
Lexus              1
BAIC YX            1
Tata               1
Chrysler          

In [309]:
# 5. Year Filtering - CLEAN VERSION
print(f"=== YEAR FILTERING ===")

# Extract year from registration_year
def extract_year_fixed(year_str):
    if pd.isna(year_str):
        return None
    year_str = str(year_str).strip()
    if '.' in year_str:
        parts = year_str.split('.')
        if len(parts) == 2:
            year_part = parts[1]
            try:
                year = int(year_part)
                if 1900 <= year <= 2030:
                    return year
            except:
                return None
    return None

merged_clean['year_extracted'] = merged_clean['registration_year'].apply(extract_year_fixed)

# Remove cars outside 2007-2025 range and null years
current_year = 2025
initial_count = len(merged_clean)
merged_clean = merged_clean[
    (merged_clean['year_extracted'] >= 2007) & 
    (merged_clean['year_extracted'] <= current_year) &
    (merged_clean['year_extracted'].notnull())
].copy()

print(f"Rows removed by year filtering: {initial_count - len(merged_clean)}")
print(f"Final dataset shape: {merged_clean.shape}")

=== YEAR FILTERING ===
Rows removed by year filtering: 35
Final dataset shape: (2336, 14)


In [310]:
# 6. Mileage Outlier Removal (for used cars only, keep new cars with 0 km)

# Convert kilometrage properly - remove 'km' and spaces, then convert to float
merged_clean['kilometrage'] = merged_clean['kilometrage'].astype(str).str.replace(' km', '').str.replace(' ', '').astype(float)

# Remove unrealistic mileages (e.g., over 500,000 km) but keep new cars (0 km)
merged_clean = merged_clean[(merged_clean['kilometrage'] <= 250000) | (merged_clean['new'] == 'yes')].copy()
print(f"After mileage filtering: {merged_clean.shape}")

# Final summary
print(f"\n=== CLEANING SUMMARY ===")
print(f"Initial rows: {len(merged_df)}")
print(f"Final rows: {len(merged_clean)}")
print(f"Rows removed: {len(merged_df) - len(merged_clean)}")
print(f"Removal percentage: {((len(merged_df) - len(merged_clean)) / len(merged_df) * 100):.1f}%")


merged_clean.to_csv('car_prices_cleaned.csv', index=False)



After mileage filtering: (2230, 14)

=== CLEANING SUMMARY ===
Initial rows: 2658
Final rows: 2230
Rows removed: 428
Removal percentage: 16.1%


In [311]:
print(merged_clean.isnull().sum())

# Check categorical variables
print("Categorical variables value counts:")
for col in ['brand', 'body_type', 'fuel_type_simple', 'transmission', 'climatisation', 'new']:
    print(f"\n{col}:")
    print(merged_clean[col].value_counts())


brand                0
model                0
price                0
kilometrage          0
body_type            0
fuel_type            0
seats                0
transmission         0
puissance_fiscale    0
registration_year    0
climatisation        0
new                  0
fuel_type_simple     0
year_extracted       0
dtype: int64
Categorical variables value counts:

brand:
Mercedes-Benz    364
Volkswagen       182
KIA              155
BMW              130
Peugeot          130
Audi             126
Hyundai           65
Land Rover        64
Ford              60
Seat              51
Toyota            50
bmw               49
GWM               49
Citroën           46
Renault           45
MG                41
mercedes-benz     34
hyundai           33
Nissan            31
Porsche           31
Jeep              31
Fiat              31
Chery             30
Suzuki            30
Mazda             26
kia               22
peugeot           20
Skoda             19
Dacia             17
mg          

In [312]:
# 1. Fix brand name inconsistencies
brand_mapping = {
    'bmw': 'BMW', 'mercedes-benz': 'Mercedes-Benz', 'hyundai': 'Hyundai',
    'kia': 'KIA', 'peugeot': 'Peugeot', 'toyota': 'Toyota', 
    'volkswagen': 'Volkswagen', 'skoda': 'Skoda', 'suzuki': 'Suzuki',
    'renault': 'Renault', 'audi': 'Audi', 'volvo': 'Volvo',
    'seat': 'Seat', 'honda': 'Honda', 'fiat': 'Fiat', 'citroen': 'Citroën',
    'porsche': 'Porsche', 'mg': 'MG'
}

merged_clean['brand'] = merged_clean['brand'].replace(brand_mapping)

# 2. Fix climatisation inconsistencies  
merged_clean['climatisation'] = merged_clean['climatisation'].replace({'No': 'no', 'Yes': 'yes'})

# 3. Remove old fuel_type and rename fuel_type_simple
merged_clean = merged_clean.drop('fuel_type', axis=1)
merged_clean = merged_clean.rename(columns={'fuel_type_simple': 'fuel_type'})

# 4. Add month column from registration_year
def extract_month(year_str):
    if pd.isna(year_str):
        return None
    year_str = str(year_str)
    if '.' in year_str:
        return int(year_str.split('.')[0])
    return None

merged_clean['month'] = merged_clean['registration_year'].apply(extract_month)
def extract_fiscal_power(value):
    if pd.isna(value):
        return None
    value_str = str(value)
    # Extract first number from string
    import re
    numbers = re.findall(r'\d+', value_str)
    if numbers:
        return float(numbers[0])
    return None

# 5. Convert puissance_fiscale to numeric (FIX FOR THE ERROR)
merged_clean['puissance_fiscale'] = merged_clean['puissance_fiscale'].apply(extract_fiscal_power)

# 6. Feature Engineering
merged_clean['car_age'] = 2025 - merged_clean['year_extracted']
merged_clean['is_new'] = (merged_clean['new'] == 'yes').astype(int)
merged_clean['price_per_fiscal'] = merged_clean['price'] / merged_clean['puissance_fiscale']

print(f"Dataset shape: {merged_clean.shape}")
print(f"New columns: month, car_age, is_new, price_per_fiscal")
print(f"Columns: {merged_clean.sample(10)}")

Dataset shape: (2230, 17)
New columns: month, car_age, is_new, price_per_fiscal
Columns:               brand        model     price  kilometrage body_type seats  \
1482            GWM     Haval H6   62000.0     211780.0       SUV     5   
1166  Mercedes-Benz          CLA  157000.0      80000.0   Berline     5   
989        Mahindra      KUV 100   27500.0     100000.0       SUV     5   
10       Volkswagen       Golf 7   69000.0     132000.0  Compacte     5   
1076        Porsche     Panamera  190000.0     140000.0   Berline     4   
2095     Volkswagen       Golf 8   82000.0     110000.0  Compacte     5   
1256     Land Rover  Range Rover  350000.0      52000.0       SUV     5   
161            Audi           Q8  350000.0      71000.0       SUV     5   
1427         Suzuki        Swift   43500.0      86000.0  Citadine     5   
807             KIA       Stonic   69000.0      91000.0       SUV     5   

     transmission  puissance_fiscale registration_year climatisation new  \
1482  Aut

In [313]:
# Convert binary categoricals to 0/1
binary_mapping = {
    'climatisation': {'yes': 1, 'no': 0},
    'new': {'yes': 1, 'no': 0},
    'transmission': {'Automatique': 1, 'Manuelle': 0}
}

for col, mapping in binary_mapping.items():
    merged_clean[col] = merged_clean[col].map(mapping)

# Verify conversions
print("Binary conversions:")
for col in ['climatisation', 'new', 'transmission']:
    print(f"{col}: {merged_clean[col].value_counts()}")

print(f"\nDataset shape: {merged_clean.shape}")
print(f"Columns: {merged_clean.sample(10)}")

Binary conversions:
climatisation: 1    1226
0    1004
Name: climatisation, dtype: int64
new: 0    1887
1     343
Name: new, dtype: int64
transmission: 1    1439
0     791
Name: transmission, dtype: int64

Dataset shape: (2230, 17)
Columns:               brand             model     price  kilometrage  body_type seats  \
1632        Peugeot               208   40000.0      45000.0   Citadine     5   
2579  Mercedes-Benz  classe-c-hybride  333000.0          0.0    Berline     5   
672          Suzuki            Ertiga   49500.0     153000.0  Monospace     7   
1591             MG                 5   38000.0     168000.0    Berline     5   
1507     Volkswagen        Polo Sedan   56000.0      74000.0    Berline     5   
2111     Volkswagen            Tiguan   59800.0     128000.0        SUV     5   
533      Volkswagen            Golf 8  138000.0      50000.0   Compacte     5   
833            Seat             Arona   79500.0      90000.0        SUV     5   
2119     Volkswagen           

In [314]:
# Let's try manual target encoding
def manual_target_encode(series, target):
    return series.map(target.groupby(series).mean())

# Apply target encoding
merged_clean['brand_encoded'] = manual_target_encode(merged_clean['brand'], merged_clean['price'])
merged_clean['body_type_encoded'] = manual_target_encode(merged_clean['body_type'], merged_clean['price'])
merged_clean['model_encoded'] = manual_target_encode(merged_clean['model'], merged_clean['price'])

# Drop original categorical columns
merged_clean = merged_clean.drop(['brand', 'body_type', 'model', 'registration_year'], axis=1)

# Print sample of the final dataframe
print("Final encoded dataset sample:")
print(merged_clean.sample(10).round(2))

print(f"\nDataset shape: {merged_clean.shape}")
print(f"Columns: {merged_clean.columns.tolist()}")

Final encoded dataset sample:
         price  kilometrage seats  transmission  puissance_fiscale  \
1221  289500.0       9000.0     5             1                9.0   
944    48000.0     250000.0     5             1               12.0   
1817  235000.0      49000.0     5             1               10.0   
2451  169900.0          0.0     9             0                8.0   
1266  114000.0      37000.0     5             1                8.0   
1347   85000.0     175000.0     5             1               10.0   
1076  190000.0     140000.0     4             1               30.0   
1098  165000.0      59764.0     5             1               10.0   
1360  129000.0     130000.0     5             1               10.0   
43    163000.0      72000.0     5             1                8.0   

      climatisation  new   fuel_type  year_extracted  month  car_age  is_new  \
1221              1    0  electrique            2024     12        1       0   
944               0    0      diesel   

In [315]:
from sklearn.preprocessing import StandardScaler

cols_to_scale = [
    'kilometrage',           
    'puissance_fiscale',    
    'car_age',              
    'price_per_fiscal',     
    'month',                
    'brand_encoded',      
    'body_type_encoded',   
    'model_encoded'        
]

scaler = StandardScaler()

# Scale the selected columns
merged_clean[cols_to_scale] = scaler.fit_transform(merged_clean[cols_to_scale])

print("Scaled dataset sample:")
print(merged_clean.sample(5).round(3))

Scaled dataset sample:
         price  kilometrage seats  transmission  puissance_fiscale  \
1417   55000.0        0.019     5             0             -0.302   
468    69000.0       -0.071     5             0             -0.541   
1060  185000.0       -0.956     5             1              0.896   
239    47000.0       -0.011     5             0             -0.541   
2214   64990.0       -1.346     2             0             -0.781   

      climatisation  new   fuel_type  year_extracted  month  car_age  is_new  \
1417              1    0     essence            2019 -0.068    0.154       0   
468               0    0     essence            2022  0.736   -0.570       0   
1060              0    0  electrique            2023 -1.675   -0.811       0   
239               1    0     essence            2019 -0.603    0.154       0   
2214              0    1      diesel            2025  1.271   -1.294       1   

      price_per_fiscal  brand_encoded  body_type_encoded  model_encoded  
1

In [316]:

# One-hot encode fuel_type
merged_clean = pd.get_dummies(merged_clean, columns=['fuel_type'], prefix='fuel')

# Remove redundant columns
columns_to_drop = [
    'year_extracted',    # Replaced by car_age
    'new',               # Replaced by is_new (0/1)
    'registration_year'  # Already extracted month/year
]

# Only drop columns that actually exist
existing_cols_to_drop = [col for col in columns_to_drop if col in merged_clean.columns]
merged_clean = merged_clean.drop(existing_cols_to_drop, axis=1)

print(f"Final dataset shape: {merged_clean.shape}")
print(f"Final columns: {merged_clean.columns.tolist()}")

print("\nFinal dataset sample:")
print(merged_clean.sample(5).round(3))

Final dataset shape: (2230, 17)
Final columns: ['price', 'kilometrage', 'seats', 'transmission', 'puissance_fiscale', 'climatisation', 'month', 'car_age', 'is_new', 'price_per_fiscal', 'brand_encoded', 'body_type_encoded', 'model_encoded', 'fuel_diesel', 'fuel_electrique', 'fuel_essence', 'fuel_hybride']

Final dataset sample:
         price  kilometrage seats  transmission  puissance_fiscale  \
614    28500.0        0.799     5             0             -0.781   
854   320000.0        0.109     5             1              0.896   
2291   86000.0       -1.346     3             0             -0.062   
1939  170000.0       -0.138     5             1             -0.062   
2486  169900.0       -1.346     5             1              0.178   

      climatisation  month  car_age  is_new  price_per_fiscal  brand_encoded  \
614               0 -0.603    0.637       0            -1.040         -0.932   
854               0  1.004   -0.087       0             1.907          1.874   
2291      

In [317]:
# Handle the 6 missing values
print("Finding missing values...")
missing_mask = merged_clean.isnull().any(axis=1)
print(f"Rows with missing values: {missing_mask.sum()}")

# Show which rows have missing values
if missing_mask.sum() > 0:
    print("\nRows with missing values:")
    print(merged_clean[missing_mask])
    
    # Remove rows with missing values (only 6 out of 2230)
    merged_clean = merged_clean.dropna()
    print(f"\nRemoved {missing_mask.sum()} rows with missing values")
    print(f"Final clean dataset: {merged_clean.shape[0]} rows, {merged_clean.shape[1]} columns")
else:
    print("No missing values found!")

#extract csv
merged_clean.to_csv('car_prices_final_preprocessed.csv', index=False)



Finding missing values...
Rows with missing values: 3

Rows with missing values:
         price  kilometrage seats  transmission  puissance_fiscale  \
2531  249900.0    -1.345653     5             1                NaN   
2553  438000.0    -1.345653     5             1                NaN   
2597  399900.0    -1.345653     5             1                NaN   

      climatisation     month   car_age  is_new  price_per_fiscal  \
2531              1  1.271464 -1.293929       1               NaN   
2553              1  1.271464 -1.293929       1               NaN   
2597              1  1.271464 -1.293929       1               NaN   

      brand_encoded  body_type_encoded  model_encoded  fuel_diesel  \
2531       1.226440           0.764927       2.001368            0   
2553       1.877001           0.764927       3.488239            0   
2597       1.052263           0.764927       3.906708            0   

      fuel_electrique  fuel_essence  fuel_hybride  
2531                1       